In [ ]:
#@title 🎧 Download Narration Audio & Play Introduction
import os as _os
if not _os.path.exists("/content/narration"):
    !pip install -q gdown
    import gdown
    gdown.download(id="1NQ4LjxgjibTul_bFVO3YyqOi8-odNDmh", output="/content/narration.zip", quiet=False)
    !unzip -q /content/narration.zip -d /content/narration
    !rm /content/narration.zip
    print(f"Loaded {len(_os.listdir('/content/narration'))} narration segments")
else:
    print("Narration audio already loaded.")

from IPython.display import Audio, display
display(Audio("/content/narration/02_00_intro.mp3"))


In [ ]:
#@title 🎧 Code Walkthrough: Setup Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_01_setup_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Setup: Run this cell first!
import torch
import sys
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("No GPU detected. This notebook runs fine on CPU.")

print(f"\nPython {sys.version.split()[0]}")
print(f"PyTorch {torch.__version__}")

import random
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Random seed set to {SEED}")

%matplotlib inline

In [ ]:
#@title 🎧 Listen: Why It Matters
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_02_why_it_matters.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


# Ring Attention from Scratch -- Vizuara

## 1. Why Does This Matter?

In the previous notebook, we established that the attention score matrix grows as $O(S^2)$, making long-context training impossible on a single GPU. We also implemented chunked attention with online softmax -- splitting K, V into chunks and processing them one at a time.

**Ring Attention** takes the same idea and distributes it across multiple GPUs. Each GPU holds one chunk of Q, K, V. The Q chunks stay fixed, and the KV chunks rotate around a logical ring. After $N$ steps (where $N$ is the number of GPUs), every GPU has seen all KV chunks and has computed the complete attention for its own queries.

In this notebook, we will:
1. **Simulate** Ring Attention with virtual GPUs on a single device
2. **Trace** the KV rotation pattern step by step
3. **Verify** that the output exactly matches standard attention
4. **Analyze** the communication-computation overlap that makes Ring Attention efficient
5. **Implement** causal masking and see why it causes load imbalance

By the end, you will have a complete working implementation of Ring Attention.

In [ ]:
#@title 🎧 Transition: Virtual Gpus
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_03_virtual_gpus.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 2. Building the Ring: Virtual GPUs

Since Colab gives us one GPU, we will simulate a ring of $N$ GPUs using Python lists. Each "GPU" is a dictionary holding its local Q, K, V chunks and its running accumulators.

In [ ]:
#@title 🎧 Code Walkthrough: Virtual Gpu Class
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_04_virtual_gpu_class.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
class VirtualGPU:
    """Simulates a single GPU in the Ring Attention setup."""

    def __init__(self, rank, Q_chunk, K_chunk, V_chunk):
        self.rank = rank
        self.Q = Q_chunk           # Fixed: never moves
        self.d_h = Q_chunk.shape[-1]
        B, S_chunk = Q_chunk.shape[0], Q_chunk.shape[1]

        # Current KV chunk (will be replaced each step)
        self.K_current = K_chunk.clone()
        self.V_current = V_chunk.clone()
        self.kv_source = rank  # Track which GPU's KV we currently hold

        # Online softmax accumulators
        self.running_max = torch.full((B, S_chunk, 1), float('-inf'),
                                      dtype=Q_chunk.dtype)
        self.running_sum = torch.zeros((B, S_chunk, 1), dtype=Q_chunk.dtype)
        self.running_out = torch.zeros_like(Q_chunk)

    def compute_partial_attention(self):
        """Compute partial attention with current KV chunk using online softmax."""
        logits = self.Q @ self.K_current.transpose(-2, -1) / (self.d_h ** 0.5)

        # Online softmax update
        chunk_max = logits.max(dim=-1, keepdim=True).values
        new_max = torch.maximum(self.running_max, chunk_max)

        alpha = torch.exp(self.running_max - new_max)
        beta = torch.exp(logits - new_max)

        self.running_sum = alpha * self.running_sum + beta.sum(dim=-1, keepdim=True)
        self.running_out = alpha * self.running_out + beta @ self.V_current
        self.running_max = new_max

    def get_output(self):
        """Return the final attention output after all steps."""
        return self.running_out / self.running_sum

    def receive_kv(self, K_new, V_new, source_rank):
        """Receive a new KV chunk (simulates network transfer)."""
        self.K_current = K_new.clone()
        self.V_current = V_new.clone()
        self.kv_source = source_rank


print("VirtualGPU class defined.")
print("Each virtual GPU holds:")
print("  - Fixed Q chunk (never moves)")
print("  - Current KV chunk (rotates each step)")
print("  - Online softmax accumulators (running_max, running_sum, running_out)")

In [ ]:
#@title 🎧 Listen: Full Algorithm Overview
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_05_full_algorithm_overview.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 3. Ring Attention: The Full Algorithm

Now let us implement the complete Ring Attention algorithm:

1. **Initialize**: Partition the sequence into $N$ chunks, one per GPU
2. **Loop for $N$ steps**: At each step, every GPU computes partial attention with its current KV chunk, then passes its KV to the next GPU in the ring
3. **Finalize**: Each GPU normalizes its accumulated output

In [ ]:
#@title 🎧 Code Walkthrough: Ring Attention Implementation
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_06_ring_attention_implementation.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def ring_attention(Q_full, K_full, V_full, num_gpus=4, verbose=True):
    """
    Simulate Ring Attention across num_gpus virtual GPUs.

    Q_full, K_full, V_full: (batch, seq_len, d_head)
    Returns: (batch, seq_len, d_head) -- identical to standard attention
    """
    B, S, d_h = Q_full.shape
    chunk_size = S // num_gpus
    assert S % num_gpus == 0, f"Sequence length {S} must be divisible by {num_gpus}"

    # Step 1: Partition sequence into chunks
    gpus = []
    for i in range(num_gpus):
        start = i * chunk_size
        end = start + chunk_size
        gpu = VirtualGPU(
            rank=i,
            Q_chunk=Q_full[:, start:end, :],
            K_chunk=K_full[:, start:end, :],
            V_chunk=V_full[:, start:end, :]
        )
        gpus.append(gpu)

    if verbose:
        print(f"Ring Attention with {num_gpus} GPUs, seq_len={S}, chunk_size={chunk_size}")
        print(f"Each GPU holds Q chunk of {chunk_size} tokens (fixed)")
        print()

    # Step 2: Ring rotation loop
    for step in range(num_gpus):
        if verbose:
            kv_map = [f"GPU{g.rank}: Q{g.rank} + KV{g.kv_source}" for g in gpus]
            print(f"Step {step + 1}/{num_gpus}: {' | '.join(kv_map)}")

        # Each GPU computes partial attention with current KV
        for gpu in gpus:
            gpu.compute_partial_attention()

        # Rotate KV chunks: GPU i sends to GPU (i+1) % N
        if step < num_gpus - 1:  # No rotation needed after last step
            # Save current KV chunks before rotation
            kv_buffer = [(gpu.K_current.clone(), gpu.V_current.clone(), gpu.kv_source)
                         for gpu in gpus]

            # Each GPU receives KV from its predecessor in the ring
            for i in range(num_gpus):
                sender = (i - 1) % num_gpus
                K_new, V_new, source = kv_buffer[sender]
                gpus[i].receive_kv(K_new, V_new, source)

    if verbose:
        print(f"\nAll {num_gpus} GPUs have processed all {num_gpus} KV chunks.")

    # Step 3: Gather outputs
    outputs = [gpu.get_output() for gpu in gpus]
    return torch.cat(outputs, dim=1)  # Reassemble full sequence


# Standard attention for reference
def standard_attention(Q, K, V):
    d_h = Q.shape[-1]
    scores = Q @ K.transpose(-2, -1) / (d_h ** 0.5)
    weights = torch.softmax(scores, dim=-1)
    return weights @ V


# Test with a small example
B, S, d_h = 1, 16, 8
torch.manual_seed(42)
Q = torch.randn(B, S, d_h)
K = torch.randn(B, S, d_h)
V = torch.randn(B, S, d_h)

out_standard = standard_attention(Q, K, V)
out_ring = ring_attention(Q, K, V, num_gpus=4, verbose=True)

diff = (out_standard - out_ring).abs().max().item()
print(f"\nMax difference from standard attention: {diff:.2e}")
print(f"Match: {'YES -- Ring Attention is exact!' if diff < 1e-5 else 'NO -- bug detected'}")

In [ ]:
#@title 🎧 Transition: Visualize Rotation Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_07_visualize_rotation_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 4. Visualizing the KV Rotation Pattern

Let us create a visual diagram showing which KV chunk each GPU processes at each step of the ring rotation.

In [ ]:
#@title 🎧 What to Look For: Visualize Rotation Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_08_visualize_rotation_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def visualize_ring_rotation(num_gpus=4):
    """Visualize which KV chunk each GPU holds at each step."""
    fig, ax = plt.subplots(figsize=(12, 5))

    colors = plt.cm.Set2(np.linspace(0, 1, num_gpus))

    cell_w, cell_h = 1.5, 0.8
    gap_x, gap_y = 0.3, 0.3

    for step in range(num_gpus):
        y = (num_gpus - 1 - step) * (cell_h + gap_y)

        # Step label
        ax.text(-1.2, y + cell_h / 2, f'Step {step + 1}',
                fontsize=12, fontweight='bold', va='center', ha='center')

        for gpu in range(num_gpus):
            x = gpu * (cell_w + gap_x)

            # Which KV chunk does this GPU hold at this step?
            kv_source = (gpu - step) % num_gpus

            # Draw cell
            rect = plt.Rectangle((x, y), cell_w, cell_h,
                                  facecolor=colors[kv_source], edgecolor='black',
                                  linewidth=1.5, alpha=0.8)
            ax.add_patch(rect)

            # Label
            ax.text(x + cell_w / 2, y + cell_h * 0.65,
                    f'Q{gpu}', fontsize=10, fontweight='bold',
                    ha='center', va='center', color='navy')
            ax.text(x + cell_w / 2, y + cell_h * 0.3,
                    f'KV{kv_source}', fontsize=10, fontweight='bold',
                    ha='center', va='center', color='darkred')

    # Column headers
    for gpu in range(num_gpus):
        x = gpu * (cell_w + gap_x)
        top_y = (num_gpus) * (cell_h + gap_y) - gap_y
        ax.text(x + cell_w / 2, top_y, f'GPU {gpu}',
                fontsize=12, fontweight='bold', ha='center', va='center')

    # Legend
    patches = [mpatches.Patch(facecolor=colors[i], edgecolor='black',
                              label=f'KV Chunk {i}') for i in range(num_gpus)]
    ax.legend(handles=patches, loc='upper right', fontsize=9, title='KV Source')

    ax.set_xlim(-2, num_gpus * (cell_w + gap_x))
    ax.set_ylim(-0.5, (num_gpus + 0.5) * (cell_h + gap_y))
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(f'Ring Attention: KV Rotation with {num_gpus} GPUs\n'
                 f'(Q stays fixed, KV rotates each step)',
                 fontsize=14, fontweight='bold', pad=15)

    plt.tight_layout()
    plt.show()

    # Print rotation summary
    print(f"\nAfter {num_gpus} steps, each GPU has seen all {num_gpus} KV chunks.")
    print("The rotation pattern: KV chunk from GPU j arrives at GPU i at step (i - j) mod N.")


visualize_ring_rotation(4)

In [ ]:
#@title 🎧 Transition: Scaling Test Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_09_scaling_test_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 5. Scaling Test: Correctness at Larger Sizes

Let us verify Ring Attention correctness with larger, more realistic tensor sizes and different GPU counts.

In [ ]:
#@title 🎧 Code Walkthrough: Scaling Test Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_10_scaling_test_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Test across multiple configurations
configs = [
    (1, 64, 32, 2),     # B=1, S=64, d_h=32, N=2
    (1, 128, 64, 4),    # B=1, S=128, d_h=64, N=4
    (2, 256, 64, 4),    # B=2, S=256, d_h=64, N=4
    (1, 512, 128, 8),   # B=1, S=512, d_h=128, N=8
    (1, 1024, 64, 4),   # B=1, S=1024, d_h=64, N=4
    (1, 2048, 128, 8),  # B=1, S=2048, d_h=128, N=8
]

print(f"{'Config':>30} | {'Max Diff':>12} | {'Status':>8}")
print("-" * 60)

for B, S, d_h, N in configs:
    torch.manual_seed(42)
    Q = torch.randn(B, S, d_h)
    K = torch.randn(B, S, d_h)
    V = torch.randn(B, S, d_h)

    out_std = standard_attention(Q, K, V)
    out_ring = ring_attention(Q, K, V, num_gpus=N, verbose=False)

    diff = (out_std - out_ring).abs().max().item()
    status = "PASS" if diff < 1e-4 else "FAIL"
    config_str = f"B={B}, S={S}, d_h={d_h}, N={N}"
    print(f"{config_str:>30} | {diff:>12.2e} | {status:>8}")

print("\nAll tests should PASS. Ring Attention is numerically exact")
print("(up to floating point rounding from online softmax).")

In [ ]:
#@title 🎧 Listen: Overlap Analysis Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_11_overlap_analysis_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 6. Communication-Computation Overlap Analysis

In real Ring Attention, the KV transfer to the next GPU happens **simultaneously** with the attention computation. Let us model when this overlap is effective.

In [ ]:
#@title 🎧 Code Walkthrough: Overlap Analysis Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_12_overlap_analysis_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def analyze_overlap(seq_lengths, num_gpus_list, d_head=128, num_heads=32,
                    bandwidth_GBps=600, gpu_tflops=312):
    """
    Analyze whether computation masks communication in Ring Attention.

    bandwidth_GBps: NVLink bandwidth (GB/s)
    gpu_tflops: GPU compute throughput (TFLOP/s FP16)
    """
    results = []

    for S in seq_lengths:
        for N in num_gpus_list:
            chunk_size = S // N

            # Communication: transfer KV chunk (2 tensors, each chunk_size x d_head, FP16)
            kv_bytes = 2 * chunk_size * d_head * 2 * num_heads  # FP16
            comm_time_ms = kv_bytes / (bandwidth_GBps * 1e9) * 1000

            # Computation: attention for one chunk
            # Per head: Q(chunk_size x d_head) @ K(chunk_size x d_head)^T
            #   = chunk_size^2 * d_head * 2 FLOPs (matmul)
            # Plus softmax + V projection: ~same order
            flops_per_head = 4 * chunk_size * chunk_size * d_head  # ~4x for full attention
            total_flops = flops_per_head * num_heads
            compute_time_ms = total_flops / (gpu_tflops * 1e12) * 1000

            overlap_ratio = compute_time_ms / comm_time_ms if comm_time_ms > 0 else float('inf')
            hidden = overlap_ratio >= 1.0

            results.append({
                'S': S, 'N': N, 'chunk': chunk_size,
                'comm_ms': comm_time_ms, 'compute_ms': compute_time_ms,
                'ratio': overlap_ratio, 'hidden': hidden
            })

    return results


seq_lengths = [8192, 32768, 65536, 131072, 524288]
num_gpus_list = [2, 4, 8, 16]

results = analyze_overlap(seq_lengths, num_gpus_list)

print(f"{'Seq Len':>8} | {'GPUs':>4} | {'Chunk':>7} | {'Compute (ms)':>12} | {'Comm (ms)':>10} | {'Comp/Comm':>10} | {'Hidden?':>8}")
print("-" * 80)

for r in results:
    print(f"{r['S']:>8,} | {r['N']:>4} | {r['chunk']:>7,} | {r['compute_ms']:>12.3f} | {r['comm_ms']:>10.3f} | {r['ratio']:>10.1f}x | {'YES' if r['hidden'] else 'NO':>8}")

print("\nWhen Comp/Comm >= 1.0, communication is fully hidden behind computation.")
print("Larger chunks = more computation per step = better overlap.")

In [ ]:
#@title 🎧 What to Look For: Overlap Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_13_overlap_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize overlap effectiveness
fig, ax = plt.subplots(figsize=(12, 6))

for N in num_gpus_list:
    rs = [r for r in results if r['N'] == N]
    chunk_sizes = [r['chunk'] for r in rs]
    ratios = [r['ratio'] for r in rs]
    ax.plot(chunk_sizes, ratios, 'o-', markersize=8, linewidth=2, label=f'N={N} GPUs')

ax.axhline(y=1.0, color='red', linestyle='--', linewidth=2, label='Overlap threshold (1.0x)')
ax.fill_between([0, max(r['chunk'] for r in results) * 1.1], 0, 1.0,
                color='red', alpha=0.1)
ax.fill_between([0, max(r['chunk'] for r in results) * 1.1], 1.0, max(r['ratio'] for r in results) * 1.2,
                color='green', alpha=0.1)

ax.text(5000, 0.5, 'Communication NOT hidden', fontsize=11, color='red', fontweight='bold')
ax.text(5000, max(r['ratio'] for r in results) * 0.8, 'Communication fully hidden',
        fontsize=11, color='green', fontweight='bold')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Chunk Size (tokens per GPU)', fontsize=13)
ax.set_ylabel('Compute / Communication Ratio', fontsize=13)
ax.set_title('Ring Attention: Communication Overlap Analysis\n'
             '(NVLink 600 GB/s, A100 312 TFLOP/s, 32 heads, d_h=128)',
             fontsize=14, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print("Key insight: As long as each GPU's chunk is large enough,")
print("the attention computation takes longer than the KV transfer,")
print("and communication is completely hidden.")

In [ ]:
#@title 🎧 Listen: Causal Masking Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_14_causal_masking_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 7. Causal Masking in Ring Attention

For autoregressive language models, token $i$ can only attend to tokens $0, 1, \ldots, i$. When we split the sequence into contiguous chunks, this creates a severe load imbalance.

In [ ]:
#@title 🎧 Code Walkthrough: Causal Attention Code
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_15_causal_attention_code.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
def ring_attention_causal(Q_full, K_full, V_full, num_gpus=4, verbose=True):
    """
    Ring Attention with causal masking (contiguous chunks).
    Returns output and per-GPU work statistics.
    """
    B, S, d_h = Q_full.shape
    chunk_size = S // num_gpus

    # Track how many non-masked logits each GPU computes
    work_per_gpu = [0] * num_gpus
    steps_active = [0] * num_gpus

    gpus = []
    for i in range(num_gpus):
        start = i * chunk_size
        end = start + chunk_size
        gpu = VirtualGPU(
            rank=i,
            Q_chunk=Q_full[:, start:end, :],
            K_chunk=K_full[:, start:end, :],
            V_chunk=V_full[:, start:end, :]
        )
        gpus.append(gpu)

    for step in range(num_gpus):
        for i, gpu in enumerate(gpus):
            # Determine which KV chunk we have
            kv_source = gpu.kv_source

            # Query positions: [i*chunk_size, (i+1)*chunk_size)
            q_start = i * chunk_size
            q_end = q_start + chunk_size

            # KV positions: [kv_source*chunk_size, (kv_source+1)*chunk_size)
            kv_start = kv_source * chunk_size
            kv_end = kv_start + chunk_size

            # Under causal masking: query at position q can only attend to KV at position <= q
            # If the entire KV chunk is AFTER all query positions, skip it
            if kv_start >= q_end:
                # All KV positions are in the future -- no work
                if verbose and step == 0:
                    pass  # Will print summary later
                continue

            # Count non-masked entries
            if kv_end <= q_start:
                # Entire KV chunk is in the past -- full attention
                nonmasked = chunk_size * chunk_size
            else:
                # Partial overlap -- triangular mask
                nonmasked = 0
                for qi in range(q_start, q_end):
                    for ki in range(kv_start, kv_end):
                        if ki <= qi:
                            nonmasked += 1

            work_per_gpu[i] += nonmasked
            steps_active[i] += 1

            # Compute partial attention with causal mask
            logits = gpu.Q @ gpu.K_current.transpose(-2, -1) / (d_h ** 0.5)

            # Apply causal mask
            for qi_local in range(chunk_size):
                qi_global = q_start + qi_local
                for ki_local in range(chunk_size):
                    ki_global = kv_start + ki_local
                    if ki_global > qi_global:
                        logits[:, qi_local, ki_local] = float('-inf')

            # Check if all entries are masked
            if logits.max() == float('-inf'):
                continue

            chunk_max = logits.max(dim=-1, keepdim=True).values
            new_max = torch.maximum(gpu.running_max, chunk_max)
            alpha = torch.exp(gpu.running_max - new_max)
            beta = torch.exp(logits - new_max)
            # Zero out -inf entries in beta
            beta = torch.where(torch.isinf(logits) & (logits < 0),
                               torch.zeros_like(beta), beta)

            gpu.running_sum = alpha * gpu.running_sum + beta.sum(dim=-1, keepdim=True)
            gpu.running_out = alpha * gpu.running_out + beta @ gpu.V_current
            gpu.running_max = new_max

        # Rotate KV
        if step < num_gpus - 1:
            kv_buffer = [(gpu.K_current.clone(), gpu.V_current.clone(), gpu.kv_source)
                         for gpu in gpus]
            for i in range(num_gpus):
                sender = (i - 1) % num_gpus
                K_new, V_new, source = kv_buffer[sender]
                gpus[i].receive_kv(K_new, V_new, source)

    outputs = [gpu.get_output() for gpu in gpus]
    return torch.cat(outputs, dim=1), work_per_gpu, steps_active


# Run causal Ring Attention and analyze load imbalance
B, S, d_h = 1, 32, 8
torch.manual_seed(42)
Q = torch.randn(B, S, d_h)
K = torch.randn(B, S, d_h)
V = torch.randn(B, S, d_h)

_, work, active_steps = ring_attention_causal(Q, K, V, num_gpus=4, verbose=True)

max_work = max(work)
print(f"\nWork Distribution (causal, contiguous chunks):")
print(f"{'GPU':>4} | {'Work (logits)':>14} | {'Relative':>10} | {'Active Steps':>13}")
print("-" * 50)
for i in range(4):
    pct = work[i] / max_work * 100 if max_work > 0 else 0
    bar = '#' * int(pct / 5)
    print(f"{i:>4} | {work[i]:>14,} | {pct:>9.1f}% | {active_steps[i]:>13} | {bar}")

avg_work = sum(work) / len(work)
utilization = avg_work / max_work * 100 if max_work > 0 else 0
print(f"\nUtilization: {utilization:.1f}% (ideal = 100%)")
print(f"Wasted: {100 - utilization:.1f}% of GPU time is idle.")

In [ ]:
#@title 🎧 What to Look For: Causal Imbalance Visualization
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_16_causal_imbalance_visualization.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# Visualize the load imbalance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart of work per GPU
colors = ['#3498db', '#e67e22', '#2ecc71', '#e74c3c']
bars = ax1.bar(range(4), [w / max_work * 100 for w in work], color=colors,
               edgecolor='black', linewidth=1.2)
ax1.set_xlabel('GPU', fontsize=13)
ax1.set_ylabel('Relative Work (%)', fontsize=13)
ax1.set_title('Causal Attention: Load Imbalance\n(Contiguous Chunks, 4 GPUs)', fontsize=13, fontweight='bold')
ax1.set_xticks(range(4))
ax1.set_xticklabels([f'GPU {i}' for i in range(4)], fontsize=11)
ax1.set_ylim(0, 120)

for bar, w in zip(bars, work):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 2,
             f'{w / max_work * 100:.0f}%', ha='center', fontsize=11, fontweight='bold')

# Right: Causal mask pattern (which chunks each GPU processes)
N = 4
mask_grid = np.zeros((N, N))
for qi in range(N):
    for kvi in range(N):
        if kvi <= qi:
            mask_grid[qi, kvi] = 1.0
        if kvi == qi:
            mask_grid[qi, kvi] = 0.5  # partial (triangular)

im = ax2.imshow(mask_grid, cmap='RdYlGn', vmin=0, vmax=1, aspect='equal')
ax2.set_xlabel('KV Chunk', fontsize=13)
ax2.set_ylabel('Query GPU', fontsize=13)
ax2.set_title('Causal Mask: Which KV Chunks Are Used\n(Green=full, Yellow=partial, Red=masked)',
              fontsize=13, fontweight='bold')
ax2.set_xticks(range(N))
ax2.set_xticklabels([f'KV {i}' for i in range(N)], fontsize=10)
ax2.set_yticks(range(N))
ax2.set_yticklabels([f'GPU {i}' for i in range(N)], fontsize=10)

# Add text annotations
for qi in range(N):
    for kvi in range(N):
        if mask_grid[qi, kvi] == 1.0:
            ax2.text(kvi, qi, 'Full', ha='center', va='center', fontsize=9, fontweight='bold')
        elif mask_grid[qi, kvi] == 0.5:
            ax2.text(kvi, qi, 'Partial', ha='center', va='center', fontsize=9)
        else:
            ax2.text(kvi, qi, 'Masked', ha='center', va='center', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

print("GPU 0 (earliest tokens) does only ~25% of the work.")
print("GPU 3 (latest tokens) does 100% -- everyone else waits for it.")
print("This motivates Striped Attention (next notebook).")

In [ ]:
#@title 🎧 Narration: Todo Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_17_todo_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 8. Your Turn

### Exercise 1: Implement Ring Attention with Different Ring Sizes

Run Ring Attention with 2, 4, 8, and 16 GPUs on a sequence of length 1024. Verify that all produce the same output as standard attention.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise1
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_18_todo_exercise1.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Test Ring Attention with different numbers of virtual GPUs

B, S, d_h = 1, 1024, 64
torch.manual_seed(123)
Q = torch.randn(B, S, d_h)
K = torch.randn(B, S, d_h)
V = torch.randn(B, S, d_h)

reference = standard_attention(Q, K, V)

gpu_counts = [2, 4, 8, 16]

for N in gpu_counts:
    # TODO: Run ring_attention with num_gpus=N and verbose=False
    # TODO: Compute max absolute difference from reference
    out = None  # REPLACE THIS LINE
    diff = 0.0  # REPLACE THIS LINE
    print(f"  N={N:>2} GPUs: chunk_size={S//N:>4}, max_diff={diff:.2e}")

print("\nAll should produce identical results to standard attention.")

In [ ]:
#@title 🎧 Listen: Todo Exercise2 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_19_todo_exercise2_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 2: Causal Utilization vs Number of GPUs

The utilization under causal masking with contiguous chunks is $(N+1)/(2N)$. Compute and plot utilization for $N = 2, 4, 8, 16, 32, 64$ GPUs.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise2
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_20_todo_exercise2.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Compute and plot causal attention utilization vs number of GPUs

N_values = [2, 4, 8, 16, 32, 64]

# TODO: For each N, compute utilization = (N+1) / (2*N)
# This is because GPU i does (i+1)/N of the work, so:
# Average work = sum_{i=0}^{N-1} (i+1)/N / N = (N+1)/(2N)
# Max work = 1.0 (GPU N-1 does full work)
# Utilization = Average / Max

utilizations = []  # TODO: compute for each N

for N in N_values:
    util = None  # REPLACE THIS LINE
    utilizations.append(util if util is not None else 0)
    print(f"  N={N:>3} GPUs: utilization = {utilizations[-1]*100:.1f}%")

# TODO: Plot utilization vs N
# Hint: plt.plot(N_values, [u*100 for u in utilizations], 'bo-')
# Add a horizontal line at 100% and at 50%
# Label axes appropriately

print("\nAs N increases, utilization approaches 50% -- half the GPUs are wasted!")
print("This is why Striped Attention was invented.")

In [ ]:
#@title 🎧 Listen: Todo Exercise3 Intro
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_21_todo_exercise3_intro.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


### Exercise 3: Multi-Head Ring Attention

Our implementation handles a single attention head. Extend it to support $H$ heads by running Ring Attention independently for each head. Verify against PyTorch's multi-head attention.

In [ ]:
#@title 🎧 Before You Start: Todo Exercise3
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_22_todo_exercise3.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


In [ ]:
# TODO: Implement multi-head Ring Attention

def multihead_ring_attention(Q_full, K_full, V_full, num_heads, num_gpus=4):
    """
    Multi-head Ring Attention.

    Q_full, K_full, V_full: (batch, seq_len, num_heads * d_head)
    Returns: (batch, seq_len, num_heads * d_head)
    """
    B, S, D = Q_full.shape
    d_head = D // num_heads

    # TODO: Reshape to (B, S, num_heads, d_head)
    # TODO: For each head, run ring_attention on the (B, S, d_head) slice
    # TODO: Concatenate results back to (B, S, num_heads * d_head)

    # Hint:
    # Q_heads = Q_full.view(B, S, num_heads, d_head)
    # For h in range(num_heads):
    #     Q_h = Q_heads[:, :, h, :]  # (B, S, d_head)
    #     ... run ring_attention ...

    pass  # REPLACE WITH YOUR IMPLEMENTATION


# Test
B, S, num_heads, d_head = 1, 256, 4, 32
D = num_heads * d_head
torch.manual_seed(42)

Q = torch.randn(B, S, D)
K = torch.randn(B, S, D)
V = torch.randn(B, S, D)

# Reference: per-head standard attention
Q_h = Q.view(B, S, num_heads, d_head)
K_h = K.view(B, S, num_heads, d_head)
V_h = V.view(B, S, num_heads, d_head)
ref_outputs = []
for h in range(num_heads):
    ref_outputs.append(standard_attention(Q_h[:,:,h,:], K_h[:,:,h,:], V_h[:,:,h,:]))
reference = torch.stack(ref_outputs, dim=2).reshape(B, S, D)

# Your implementation
output = multihead_ring_attention(Q, K, V, num_heads=num_heads, num_gpus=4)

if output is not None:
    diff = (reference - output).abs().max().item()
    print(f"Multi-head Ring Attention: max diff = {diff:.2e}")
    print(f"Match: {'YES' if diff < 1e-4 else 'NO'}")
else:
    print("Complete the implementation above.")

In [ ]:
#@title 🎧 Wrap-Up: Closing
from IPython.display import Audio, display
import os as _os
_f = "/content/narration/02_23_closing.mp3"
if _os.path.exists(_f):
    display(Audio(_f))
else:
    print("Run the first cell to download narration audio.")


## 9. Summary

In this notebook we built Ring Attention from the ground up:

1. **Virtual GPU simulation**: Each GPU holds a fixed Q chunk and a rotating KV chunk, with online softmax accumulators.

2. **Ring rotation**: After $N$ steps, every GPU has seen all $N$ KV chunks. The output is **numerically identical** to standard attention.

3. **Communication overlap**: On real hardware, KV transfers happen via NVLink while attention is being computed on the GPU cores -- they operate on independent hardware and can run in parallel.

4. **Causal masking problem**: With contiguous chunks, early-token GPUs do far less work than late-token GPUs, leading to ~62.5% utilization with 4 GPUs.

In the next notebook, we will implement **Striped Attention** -- a round-robin token assignment that restores near-100% utilization under causal masking.

In [ ]:
print("=" * 60)
print("SUMMARY: Ring Attention")
print("=" * 60)
print()
print("Key results:")
print("  - Ring Attention: Q stays fixed, KV rotates around the ring")
print("  - N steps for N GPUs -- each GPU sees all KV chunks")
print("  - Online softmax ensures numerically exact results")
print("  - Communication hidden behind computation (NVLink || GPU compute)")
print("  - Causal masking + contiguous chunks = ~62.5% utilization (4 GPUs)")
print()
print("Next: Striped Attention for balanced causal masking.")